# Google Cloud Run への Streamlit チャットボットアプリのデプロイ

このガイドでは、Streamlit を使用して Gemini を活用したシンプルなチャットボットアプリケーションを構築するためのスキャフォールド（骨組み）と手順を説明します。


### Streamlit とは？
[Streamlit](https://streamlit.io/) は、データサイエンスや機械学習のプロジェクト向けに対話型の Web アプリケーションを迅速に作成するための代表的なオープンソース Python ライブラリです。Streamlit の直感的な設計により、最小限のコードで強力なアプリを簡単に構築できます。

作成可能な幅広いアプリケーションを確認するには、[Streamlit Gallery](https://streamlit.io/gallery) の素晴らしい例をご覧ください。データダッシュボードや可視化から、機械学習のデモやインタラクティブツールまで多岐にわたります！

## セットアップ

Google Cloud と連携するための変数をいくつか設定します。

In [ ]:
import os

PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
REGION = "us-central1"

os.environ["PROJECT"] = PROJECT
os.environ["REGION"] = REGION

---
## Streamlit アプリケーションの構築

アプリケーションのコードは [app.py](./app.py) に記述されています。ここでは、コード内の重要な概念を確認していきましょう。

### テキスト要素の追加
<div class="alert alert-info">

```python
st.set_page_config(page_title="Chat with Gemini", page_icon="♊")

st.title("Chat with Gemini")

st.markdown("Welcome to this simple web application to chat with Gemini")
```
</div>

Streamlit は最終的に Web ページをレンダリングしますが、Python モジュールを使用するだけでその見た目と動作を定義できます。

まず、アプリのメタデータをいくつか設定します。

- [`st.set_page_config`](https://docs.streamlit.io/develop/api-reference/configuration/st.set_page_config) を使用すると、ページタイトルやファビコン（ブラウザのタブに表示される小さなアイコン）などをカスタマイズできます。

Streamlit にはテキストコンテンツを表示するためのさまざまな方法が用意されています:

- [`st.title`](https://docs.streamlit.io/develop/api-reference/text/st.title): ページにメインの見出しを追加します。
- [`st.header`](https://docs.streamlit.io/develop/api-reference/text/st.header) や [`st.subheader`](https://docs.streamlit.io/develop/api-reference/text/st.subheader): より小さな見出しを追加できます。
- [`st.text`](https://docs.streamlit.io/develop/api-reference/text/st.text): プレーンテキストを表示します。
- [`st.markdown`](https://docs.streamlit.io/develop/api-reference/text/st.markdown): Markdown 構文を使用して書式付きテキストを追加できます。
その他のオプションについては、Streamlit ドキュメントの [他のテキスト要素](https://docs.streamlit.io/develop/api-reference/text) を確認してください。

Streamlit には万能ナイフ（スイスアーミーナイフ）のようなコマンドである [`st.write`](https://docs.streamlit.io/develop/api-reference/write-magic/st.write) も用意されています。テキスト、DataFrame（データテーブル）、Matplotlib のプロット、さらには Keras の機械学習モデルなど、さまざまな種類のコンテンツを扱うことができます。

次に、いくつかの変数を設定しましょう。

環境変数（`GCP_PROJECT` および `GCP_REGION`）は、後でアプリを Cloud Run にデプロイする際に設定できます。

<div class="alert alert-info">

```python
PROJECT_ID = os.environ.get("GCP_PROJECT")

client = genai.Client(project=PROJECT_ID, vertexai=True, location="global")

if "gemini_model" not in st.session_state:
    st.session_state["gemini_model"] = "gemini-3.5-flash"
```
</div>

### セッション状態（Session State）のセットアップ
Streamlit は、イベントが発生するたびにコードを上から下まで再実行することを覚えておいてください。<br>
[Session State](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) は、各ユーザーセッションにおいてこれら再実行間で変数を共有する方法です。`st.session_state` にキーと値の形式で値を保存できます。

これはチャットボットアプリなので、チャット履歴を保持する必要があります。`"messages"` キーを作成し、履歴をリストとして保存しましょう。<br>
セッション状態に `"messages"` キーが存在しない場合（最初にページを開いたとき）、リストを初期化します。キーがすでに作成されている場合は、リストを反復処理して各メッセージをページに表示します。

チャットボットアプリでは、[`st.chat_message`](https://docs.streamlit.io/develop/api-reference/chat/st.chat_message) コンテナを使用して各メッセージをページ上に表示できます。<br>
`with` 構文を使用して返されたコンテナに要素を追加するか、返されたオブジェクトのメソッドを直接呼び出すことができます。
例:

<div class="alert alert-info">

```python
for message in st.session_state.messages:
    with st.chat_message(name=message["role"], avatar=message["avatar"]):
        st.markdown(message["content"])
```
または
```python
for message in st.session_state.messages:
    message = st.chat_message(name=message["role"], avatar=message["avatar"])
    message.markdown(message["content"])
```
</div>

chat_message は 2 つの引数を取ります:
- `name`: メッセージ作成者の名前。プリセットのスタイルやアバターを有効にするために、"human"/"user" または "ai"/"assistant" を指定できます。
- `avatar`: メッセージの横に表示されるアバター。
  - [st.image](https://docs.streamlit.io/develop/api-reference/media/st.image) でサポートされている画像を指定できます。
  - 一部の絵文字フォーマットもサポートされています。詳細は[ドキュメント](https://docs.streamlit.io/develop/api-reference/chat/st.chat_message)を参照してください。
  - None（デフォルト）の場合、アイコンは name（"user"/"human" または "ai"/"assistant"）によって決定されます。
  
これらの引数は、後で追加する各メッセージ辞書から渡します。

### 会話のためのヘルパー関数
後で使用するヘルパー関数を定義しましょう。

`generate_response` 関数は、セッション内の履歴を使用して Gemini とのチャットを開始します。

`stream` 関数は、後ですべてのレスポンスを一度に書き込むのではなく、より対話的なユーザー体験のために Gemini のレスポンスをストリーミングで書き込むために使用されます。

<div class="alert alert-info">

```python
def generate_response(input_text):
    chat = client.chats.create(
        model=st.session_state["gemini_model"],
        history=[
            Content(role=message["role"], parts=[Part.from_text(text=message["content"])])
            for message in st.session_state.messages[:-1]
        ]
    )
    return chat.send_message(input_text)


def stream(text):
    for word in text.split(" "):
        yield word + " "
        time.sleep(0.02)
```
</div>

### チャットの反復処理の定義

チャットのやり取りの反復処理を定義しましょう。これには次のステップがあります:
1. ページ上にユーザー入力を表示します。`st.chat_message` を `"user"` 名として使用し、`st.write` を使用してメッセージを追加します。
2. セッション状態のメッセージ履歴にユーザー入力を追加します。
3. Gemini を呼び出してレスポンスを書き込みます。ここでは名前として `"assistant"` を使用し、アバターとして Gemini アイコンを指定します。また、レスポンスをストリームで表示するために `stream` 関数とともに `st.write_stream` を使用します。
4. セッション状態のメッセージ履歴に Gemini のレスポンスを追加します。

<div class="alert alert-info">

```python
if prompt := st.chat_input("Write a promt"):
    # 1. ユーザーメッセージを書き込む
    with st.chat_message(name="user", avatar=None):
        st.write(prompt)
    # 2. メッセージ履歴にユーザーメッセージを追加
    st.session_state.messages.append(
        {"role": "user", "content": prompt, "avatar": None}
    )

    # 3. Gemini を呼び出してレスポンスを書き込む
    with st.chat_message(name="assistant", avatar="assets/gemini-icon.png"):
        response = generate_response(prompt)
        st.write_stream(stream(response.text))
    # 4. メッセージ履歴に Gemini のレスポンスを追加
    st.session_state.messages.append(
        {
            "role": "model",
            "content": response.text,
            "avatar": "assets/gemini-icon.png",
        }
    )
```
</div>

---
## Cloud Run へのアプリのデプロイ
Streamlit アプリケーションの準備が整いました！コンテナ化されたアプリケーションをシームレスに実行できるように設計されたサーバーレスプラットフォームである Google Cloud Run にデプロイしましょう。

### Dockerfile と依存関係の定義
Cloud Run 向けにアプリをコンテナ化するために、`requirements.txt` と `Dockerfile` を定義します。

In [ ]:
%%writefile requirements.txt
streamlit==1.56.0
google-cloud-aiplatform==1.145.0
google-genai==1.69.0

In [ ]:
%%writefile Dockerfile
FROM python:3.12

WORKDIR /app

COPY requirements.txt /app
RUN pip install -r requirements.txt

COPY assets /app/assets
COPY app.py /app

EXPOSE 8080

CMD streamlit run --server.port 8080 --server.enableCORS false app.py

**注: `COPY` コマンドを複数行に分割し、それぞれ異なるファイルをコピーしています。これは必須ではありませんが、Docker のキャッシュメカニズムにとって非常に重要な最適化です。<br> app.py のみを変更した場合、次回イメージをビルドする際に Docker は依存関係のインストールやその他のファイルのキャッシュされたレイヤーを再利用するため、ビルドプロセスが大幅に高速化されます。**

### コンテナのビルドと Artifact Registry への push
`Dockerfile` が準備できたので、Streamlit アプリの Docker イメージをビルドし、Google Cloud の Artifact Registry に push できます。
Artifact Registry は、コンテナイメージを保存するための安全でスケーラブルな方法を提供します。

まず、コンテナイメージを格納するために Artifact Registry に新しいリポジトリを作成します。

In [ ]:
STREAMLIT_ARTIFACT_REG_REPO = "gemini-chatbot-app"
os.environ["STREAMLIT_ARTIFACT_REG_REPO"] = STREAMLIT_ARTIFACT_REG_REPO

In [ ]:
%%bash
if ! gcloud artifacts repositories describe $STREAMLIT_ARTIFACT_REG_REPO \
       --location=$REGION > /dev/null 2>&1; then
    gcloud artifacts repositories create $STREAMLIT_ARTIFACT_REG_REPO \
        --project=$PROJECT --location=$REGION --repository-format=docker
fi

### Cloud Build 用の cloudbuild.yaml の定義
Google Cloud Build を使用して、Docker イメージのビルドと Artifact Registry への push のプロセスを自動化します。

Cloud Build は、cloudbuild.yaml という構成ファイルでビルドステップを定義できるサーバーレス CI/CD プラットフォームです。

In [ ]:
CONTAINER_PATH = (
    f"us-central1-docker.pkg.dev/{PROJECT}/{STREAMLIT_ARTIFACT_REG_REPO}/app"
)
os.environ["CONTAINER_PATH"] = CONTAINER_PATH

以下は、ビルドを高速化するためにキャッシュを取り入れた cloudbuild.yaml ファイルです:

1. **既存イメージの pull**: 最初のステップでは、Artifact Registry から Docker イメージの最新バージョンの pull を試みます。ここではエントリーポイントとして `bash -c` コマンドを使用しているため、初回の実行でこのステップが失敗してもジョブは無視して続行できます。
2. **キャッシュを使用したビルド**: 2 番目のステップでは、新しいイメージをビルドします。`--cache-from` フラグは、pull したイメージのレイヤーをキャッシュとして使用するよう Docker に指示し、それらのレイヤーに変更がない場合はビルドを高速化します。

In [ ]:
%%writefile cloudbuild.yaml
steps:
- name: 'gcr.io/cloud-builders/docker'
  entrypoint: 'bash'
  args: ['-c', 'docker pull ${_CONTAINER_PATH}:latest || exit 0']
- name: 'gcr.io/cloud-builders/docker'
  args: [
            'build',
            '-t', '${_CONTAINER_PATH}:latest',
            '--cache-from', '${_CONTAINER_PATH}:latest',
            '.'
        ]
images: ['${_CONTAINER_PATH}:latest']

### コンテナイメージのビルド

cloudbuild.yaml ファイルが定義されたので、Cloud Build に Dockerイメージのビルドを指示できます。

In [ ]:
!gcloud builds submit --config cloudbuild.yaml --region $REGION . --substitutions _CONTAINER_PATH={CONTAINER_PATH}

### Cloud Run へのデプロイ
Artifact Registry に保存されたコンテナイメージの準備ができたので、Streamlit アプリを Cloud Run にデプロイする準備がすべて整いました。

このコマンドを上記で定義した `cloudbuild.yaml` に組み込むことも検討できます。

In [ ]:
APP_NAME = "gemini-chatbot-app"
os.environ["APP_NAME"] = APP_NAME

In [ ]:
%%bash
echo 'Deploying the application to Cloud Run...'
gcloud run deploy $APP_NAME \
  --image $CONTAINER_PATH:latest --min-instances 1 --max-instances 1 --cpu 1 \
  --memory 4Gi --region us-central1 \
  --update-env-vars GCP_PROJECT=$PROJECT,GCP_REGION=$REGION > /dev/null 2>&1 && \
echo 'Deployment Done.'

### Cloud Shell 経由での Cloud Run アプリへの接続


Cloud Run サービスへのアクセス構成に関しては、非常に高い柔軟性があります。[一般公開（パブリックアクセス可能）にする](https://cloud.google.com/run/docs/authenticating/public) ことも可能です。

ただし、この例ではプロキシを使用して Cloud Shell から Cloud Run アプリに安全に接続する方法を見てみましょう。

Cloud Shell からアプリを開くには、次の手順に従います。
1. 次のセルを実行し、出力された `gcloud run services proxy ...` コマンドをコピーします。
2. Cloud Shell を開き、コマンドを貼り付けて実行します。
3. Cloud Shell のツールバーにある「ウェブでプレビュー」ボタンをクリックします。
4. 「ポート 8080 でプレビュー」を選択します。
5. 新しいブラウザタブまたはウィンドウが開き、Streamlit アプリが表示されます。

In [ ]:
print(
    f"gcloud run services proxy {APP_NAME} --project {PROJECT} --region {REGION}"
)

## アプリを試す
これでアプリをテストする準備が整いました。

Gemini との会話をお楽しみください！

<img width="600" alt="image" src="https://github.com/user-attachments/assets/08a6421a-3b50-413d-a921-55476b03b1ec">

Copyright 2024 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.